---
image: example.gif
pub-info:
    abstract: |
        A simple queueing model built with Ciw rather than SimPy, showing how to convert Ciw's own
        simulation records into a vidigi-compatible event log. Based on a published reproducible Ciw
        model example from Monks, Harper & Heather (2023).
execute: 
  enabled: true
---

# A ciw 2.x example

:::{.callout-warning}
Note that this example is written using ciw 2.x

It will not run with 3.x - but could theoretically be adapted to do so
:::

---

The underlying model code is from Monks, T., Harper, A., & Heather, A. (2023). Towards Sharing Tools, Artefacts, and Reproducible Simulation: a ciw model example (v1.0.1). Zenodo. https://doi.org/10.5281/zenodo.10051494

See here for the adaptation embedded within that repo: [https://github.com/Bergam0t/ciw-example-animation/tree/main](https://github.com/Bergam0t/ciw-example-animation/tree/main)

---

In SimPy models, we have to manually add our event logs at various points. However, for Ciw models, we instead can make use of the `event_log_from_ciw_recs` helper function from `vidigi.utils` to automatically reshape the logs ciw generates into the correct format for vidigi to work with. 

Let's start by running the model and viewing the logs ciw outputs. 

In [ ]:
# Import the wrapper objects for model interaction.
from ex_4_ciw_model import Experiment, multiple_replications

from vidigi.ciw import event_logger_from_ciw_recs, trial_logger_from_ciw_recs
from vidigi.utils import EventPosition, create_event_position_df

In [ ]:
#| echo: false
import os
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "ex_4_ciw_model.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code for the ciw model

```python
{code_content}
```

:::

""")

In [ ]:
N_OPERATORS = 18
N_NURSES = 9
RESULTS_COLLECTION_PERIOD = 1000

user_experiment = Experiment(
    n_operators=N_OPERATORS, n_nurses=N_NURSES, chance_callback=0.4
)

# run multiple replications
results, logs = multiple_replications(user_experiment, n_reps=10)

Here, the 'logs' object is the result of running `sim_engine.get_all_records()`

However, note that while we run multiple replications, we only pass the records for a single replication to the 
`event_log_from_ciw_recs` function. 

While we've done multiple replications, for the purpose of the animation we want only a single set of logs, so we will extract those from the logs variable we created. 

In [ ]:
# the 'logs' object contains a list, where each entry is the recs object for that run
logs_run_1 = logs[0]

print(len(logs_run_1))

Let's look at the first row of our result. What do we have? 

In [ ]:
logs[0][0]

Let's print all of the outputs for a single individual.

In [ ]:
[print(log) for log in logs_run_1 if log.id_number == 500]

It looks like we get one entry per node that is visited. 

Let's now use some functions vidigi provides to turn it into a format that can be used for our animation. 

### The \_from_ciw_recs family of functions

:::{.callout-tip}
Prior to vidigi 2.0.0, we would have made use of the `event_log_from_ciw_recs` helper function from `vidigi.utils` to automatically reshape ciw logs into the correct format for vidigi to work with. This would return the ciw logs as a dataframe in the format that vidigi can use. However, from v2.0.0, the recommendation has changed to using `event_logger_from_ciw_recs` returning a vidigi EventLogger object, which opens up a wide range of additional visualisations and works more comfortably with the vidigi animation functions. v2.0.0 also introduced the new `trial_log_from_ciw_recs`, which we will explore as well. 
:::

First, we'll have a go with event_logger_from_ciw_recs. We earlier imported this from the `ciw` submodule of vidigi with `from vidigi.ciw import event_logger_from_ciw_recs`. 

For each node, we must pass in an appropriate name. Vidigi will use these and append '_begins' and '_ends', as well as calculating arrivals and departures from the model and creating resource IDs to allow it to correctly show the utilisation of a resource at each step. 

In [ ]:
# let's now try turning this into an event log
event_log_test = event_logger_from_ciw_recs(
    logs_run_1, node_name_list=["operator", "nurse"]
)

event_log_test

Let's use the `.to_dataframe()` method of our event logger to take a peek at the created dataframe.  

In [ ]:
event_log_test.to_dataframe().head(25)

Let's look at using `TrialLogger`.

In [ ]:
trial_log = trial_logger_from_ciw_recs(
    logs, node_name_list=["operator", "nurse"]
)

In [ ]:
trial_log

In [ ]:
trial_log.get_log_by_run(run=5, as_df=True).head(25)

Using the `TrialLogger` class gets us access to a wide range of convenience plots and metrics in addition to the animation we will create shortly. 

In [ ]:
trial_log.plot_queue_size(
    event_list=["operator_wait_begins"],
    limit_duration=RESULTS_COLLECTION_PERIOD,
    every_x_time_units=10
    )

Now we need to create a suitable class to pass in the resource numbers to the animation function.

Like with SimPy, we need to tell vidigi where to put each step on our plot. We will refer to the names we used - so as we named our nodes 'operator' and 'nurse', we will want

- arrival
- operator_wait_begins (to show queueing for the operator)
- operator_begins (to show resource use of the operator)
- nurse_wait_begins (to show queuing for the nurse after finishing being seen by the operator)
- nurse_begins (to show resource use of the nurse)

For the _begins steps, which relat to resource use, we will also pass in a name that relates to the number of resources we need, which we defined in our model_params class above.

So, for the operator_begins step, for example, we tell ut to look for n_operators, whch is one of the parameters in our model_params class. We pass the params class into the animation function. 

In [ ]:
# Create required event_position_df for vidigi animation
event_position_df = create_event_position_df(
    [
        EventPosition(
            event="operator_wait_begins", x=205, y=270, label="Waiting for Operator"
        ),
        EventPosition(
            event="operator_begins",
            x=210,
            y=210,
            resource="n_operators",
            label="Speaking to Operator",
        ),
        EventPosition(
            event="nurse_wait_begins", x=205, y=110, label="Waiting for Nurse"
        ),
        EventPosition(
            event="nurse_begins",
            x=210,
            y=50,
            resource="n_nurses",
            label="Speaking to Nurse",
        ),
        EventPosition(event="depart", x=270, y=10, label="Exit"),
    ]
)

event_position_df

Finally, we can create the animation. 

We can access the main animation function `animate_activity_log` directly from our `TrialLogger` object (or an `EventLogger` object if we'd just done a single run). 

We will pass in our resource counts as a dict to the `scenario` parameter.

In [ ]:
trial_log.animate_activity_log(
    run_number=5,
    event_position_df=event_position_df,
    scenario={'n_operators':N_OPERATORS, 'n_nurses': N_NURSES},
    debug_mode=True,
    setup_mode=False,
    every_x_time_units=1,
    include_play_button=True,
    entity_icon_size=20,
    gap_between_entities=8,
    gap_between_queue_rows=25,
    plotly_height=700,
    frame_duration=200,
    plotly_width=1200,
    override_x_max=300,
    override_y_max=300,
    limit_duration=RESULTS_COLLECTION_PERIOD,
    wrap_queues_at=25,
    wrap_resources_at=50,
    step_snapshot_max=75,
    time_display_units="dhm",
    display_stage_labels=True,
)